# Orchestrating the Complete Claim Pipeline

This notebook is the **orchestration fundamentals** notebook. It assumes you
have already worked through `03_tool_calling_notebook.ipynb`, which defined:

- All the `[TOOL]` functions (deterministic)
- `run_react_agent` (the hand-rolled ReAct loop) and the three `[AGENT]`
  wrappers: `extract_diagnosis_and_treatment`,
  `check_diagnosis_treatment_consistency`, `check_eligibility`

Here we write the **controller** — plain Python, no LLM — that decides, at
each stage of the pipeline, whether to call a tool directly or hand off to
an agent, and that implements the *branching logic* real systems need:
what happens if the doctor hasn't signed off yet? What if the agent isn't
confident? What if the bill exceeds what's eligible?


In [ ]:
from google.colab import files

uploaded = files.upload()

## 1. The orchestration pattern

Every step below is annotated with whether it's a **direct function call**
or an **agent invocation**, matching the classification in
`01_case_study_and_functions.md`. The orchestrator's job is to:

1. Call the right thing (tool vs. agent) at the right time.
2. Pass the output of one step as the input to the next.
3. Branch: stop and escalate (`flag_for_manual_review`) instead of
   ploughing ahead when something is missing or the agent isn't confident.
4. Assemble a final, structured result.

This is intentionally a plain sequential pipeline with a few `if` branches —
not a generic "agent decides everything" loop — because most of this
workflow genuinely has a fixed, known order. Reserving free-form agent
autonomy for only the steps that need it is itself a design lesson.


In [2]:
def orchestrate_claim_process(patient_id: str, verbose: bool = True) -> dict:
    """[CONTROLLER] Run the full discharge-to-claim pipeline for one patient.

    Returns a dict describing the outcome: either a completed claim + bill,
    or a manual-review flag with a reason.
    """
    def log(msg):
        if verbose:
            print(msg)

    # --- Step 1: admission record --------------------------------- [TOOL]
    log("STEP 1 [TOOL] get_patient_record")
    patient_record = get_patient_record(patient_id)
    log(f"  -> {patient_record}")

    # --- Step 2: doctor sign-off check ------------------------------ [TOOL]
    log("STEP 2 [TOOL] get_doctor_signoff_status")
    signoff = get_doctor_signoff_status(patient_id)
    log(f"  -> {signoff}")
    if not signoff.get("signed_off"):
        reason = "Doctor has not signed off on this patient's record yet."
        log(f"  BRANCH: {reason} -> escalating.")
        return flag_for_manual_review(patient_id, reason)

    # --- Step 3: doctor's notes -------------------------------------- [TOOL]
    log("STEP 3 [TOOL] get_doctor_notes")
    notes = get_doctor_notes(patient_id)
    log(f"  -> {notes[:80]}...")

    # --- Step 4: extract diagnosis & treatment ----------------------- [AGENT]
    log("STEP 4 [AGENT] extract_diagnosis_and_treatment")
    extraction = extract_diagnosis_and_treatment(notes)
    log(f"  -> {extraction}")
    if not extraction.get("icd_code") or not extraction.get("cpt_code"):
        reason = f"Could not confidently extract/validate codes: {extraction}"
        log(f"  BRANCH: {reason} -> escalating.")
        return flag_for_manual_review(patient_id, reason)

    diagnosis_code = extraction["icd_code"]
    procedure_code = extraction["cpt_code"]

    # --- Step 5: clinical consistency check -------------------------- [AGENT]
    log("STEP 5 [AGENT] check_diagnosis_treatment_consistency")
    consistency = check_diagnosis_treatment_consistency(
        extraction["diagnosis"], extraction["treatment"]
    )
    log(f"  -> {consistency}")
    if consistency.get("needs_human_review") or not consistency.get("consistent"):
        reason = f"Diagnosis/treatment consistency check flagged this: {consistency}"
        log(f"  BRANCH: {reason} -> escalating.")
        return flag_for_manual_review(patient_id, reason)

    # --- Step 6: insurance policy ------------------------------------- [TOOL]
    log("STEP 6 [TOOL] get_insurance_policy")
    policy = get_insurance_policy(patient_record["insurance_id"])
    log(f"  -> {policy}")

    # --- Step 7: eligibility check -------------------------------------- [AGENT]
    log("STEP 7 [AGENT] check_eligibility")
    eligibility = check_eligibility(
        policy, diagnosis_code, procedure_code, patient_record["days_admitted"]
    )
    log(f"  -> {eligibility}")
    if not eligibility.get("eligible"):
        reason = f"Claim is not eligible under this policy: {eligibility.get('reasoning')}"
        log(f"  BRANCH: {reason} -> escalating.")
        return flag_for_manual_review(patient_id, reason)

    # --- Step 8: hospital bill ------------------------------------------ [TOOL]
    log("STEP 8 [TOOL] get_hospital_bill_items + calculate_total_bill")
    bill_items = get_hospital_bill_items(patient_id)
    total_bill = calculate_total_bill(bill_items)
    log(f"  -> total_bill = {total_bill}")

    # --- Step 9: money math ---------------------------------------------- [TOOL]
    log("STEP 9 [TOOL] calculate_covered_amount + calculate_patient_payable")
    covered_amount = calculate_covered_amount(
        policy, total_bill, patient_record["days_admitted"], eligibility
    )
    patient_payable = calculate_patient_payable(total_bill, covered_amount)
    log(f"  -> covered_amount = {covered_amount}, patient_payable = {patient_payable}")

    # --- Step 10: generate outputs ---------------------------------------- [TOOL]
    log("STEP 10 [TOOL] generate_claim_form + generate_discharge_bill")
    claim_form = generate_claim_form(
        patient_record, diagnosis_code, procedure_code, covered_amount,
        eligibility.get("reasoning", ""),
    )
    discharge_bill = generate_discharge_bill(
        patient_record, total_bill, covered_amount, patient_payable
    )

    return {
        "status": "completed",
        "patient_record": patient_record,
        "diagnosis": extraction["diagnosis"],
        "treatment": extraction["treatment"],
        "diagnosis_code": diagnosis_code,
        "procedure_code": procedure_code,
        "eligibility": eligibility,
        "total_bill": total_bill,
        "covered_amount": covered_amount,
        "patient_payable": patient_payable,
        "claim_form": claim_form,
        "discharge_bill": discharge_bill,
    }


## 2. Run it end to end

This mirrors the worked example in `02_react_transcript.md`, but now the
[AGENT] steps are real LLM calls (via `run_react_agent` from Notebook 1)
rather than a hand-written transcript.


In [ ]:
result = orchestrate_claim_process("P1042")
print(result)

## 3. A patient whose claim should be escalated

`P2091`'s bill (a coronary angioplasty) is far larger. Try it and see how
the eligibility cap and covered-amount math behave once the total bill
exceeds the eligible amount — this branch needs *no* extra LLM reasoning,
because `calculate_covered_amount` already knows how to cap deterministically
once eligibility has been decided.


In [ ]:
result_2091 = orchestrate_claim_process("P2091")
result_2091


## 4. Forcing an escalation path

To see the manual-review branch fire, temporarily edit `DOCTOR_SIGNOFF` (in
Notebook 1's mock data) so a patient's sign-off is `False`, then re-run
`orchestrate_claim_process` for that patient. Confirm the pipeline stops at
Step 2 and never wastes an LLM call on Steps 4, 5, or 7 — this is the
benefit of putting the cheap, deterministic checks *before* the expensive,
LLM-backed ones in the pipeline order.


In [ ]:
# Temporarily make the doctor sign-off invalid
DOCTOR_SIGNOFF["P1042"] = {
    "signed_off": False,
    "signed_by": None,
    "timestamp": None
}

# Run the claim pipeline
result = orchestrate_claim_process("P1042")

print(result)

## 5. Design takeaways for students

1. **Order matters for cost and speed.** Cheap deterministic checks
   (sign-off, record lookups) run before expensive LLM-backed reasoning
   steps, so failures are caught early and cheaply.
2. **The orchestrator is not itself an agent.** It's a fixed, auditable
   sequence of steps with explicit branches — easy to test, log, and debug.
   Only the three genuinely ambiguous sub-steps are delegated to an LLM
   ReAct loop.
3. **Escalation is a first-class outcome**, not an error. A well-designed
   pipeline for a regulated domain like insurance claims should default to
   *asking a human* whenever an agent's confidence, or a policy's clarity,
   is in question — not to silently guess.
4. **Determinism after judgment.** Once eligibility (a judgment call) is
   decided, everything downstream (money math, document generation) is
   deterministic and reproducible — which matters a lot when the output is
   a legal/financial document like an insurance claim.

## 6. Exercises

1. Add a Step 0 that checks `days_admitted <= 0` or a missing
   `insurance_id` and escalates immediately, before even checking sign-off.
2. Add a retry: if `extract_diagnosis_and_treatment` fails to resolve a code
   on the first try, re-run it once with an augmented prompt asking the
   model to consider common misspellings/synonyms, before escalating.
3. Instrument the orchestrator to also record, for each step, whether it
   was a `[TOOL]` or `[AGENT]` call and how long it took — then print a
   short summary table at the end of a run. This is the first building
   block of an observability/tracing layer for agentic pipelines.
4. Batch-run `orchestrate_claim_process` over all patient IDs in `PATIENTS`
   and produce a summary: how many completed straight through vs. how many
   were escalated, and why.


In [ ]:
import time
import pandas as pd

def run_traced(step_name, call_type, func, *args, **kwargs):
    """Run one pipeline step and record its type, duration, and outcome."""
    start = time.perf_counter()

    try:
        result = func(*args, **kwargs)
        outcome = "OK"
        return result
    except Exception as exc:
        outcome = f"ERROR: {type(exc).__name__}: {exc}"
        raise
    finally:
        elapsed = time.perf_counter() - start
        trace = {
            "step": step_name,
            "type": call_type,
            "duration_sec": round(elapsed, 4),
            "outcome": outcome,
        }
        # The active trace is set by the orchestrator.
        ACTIVE_TRACE.append(trace)


def print_trace_summary(trace):
    if not trace:
        print("No pipeline steps were recorded.")
        return

    df = pd.DataFrame(trace)
    print("\nPipeline trace")
    print(df.to_string(index=False))

    total = df["duration_sec"].sum()
    print(f"\nRecorded steps: {len(df)} | Total time: {total:.4f}s")

In [ ]:
def validate_patient_inputs(patient_record):
    """Return an error reason, or None when basic inputs are valid."""
    days = patient_record.get("days_admitted")
    insurance_id = patient_record.get("insurance_id")

    try:
        if days is None or float(days) <= 0:
            return "Invalid or missing days_admitted"
    except (TypeError, ValueError):
        return "Invalid or missing days_admitted"

    if not insurance_id or not str(insurance_id).strip():
        return "Missing insurance_id"

    return None


def extract_with_one_retry(notes):
    """Retry once if ICD/CPT code resolution is incomplete."""
    first = extract_diagnosis_and_treatment(notes)

    if first.get("icd_code") and first.get("cpt_code"):
        return first

    augmented_notes = f"""
    Resolve the diagnosis and treatment from the notes below.
    On this retry, carefully consider common misspellings, abbreviations,
    alternate clinical names, and synonyms. Normalize the wording before
    looking up the corresponding valid ICD and procedure codes.
    Do not invent codes. If a code cannot be resolved, leave it missing.

    Doctor's notes:
    {notes}
    """

    second = extract_diagnosis_and_treatment(augmented_notes)

    # Keep the second result; if it still lacks a code, the caller escalates.
    return second

In [ ]:
patient_record = run_traced(
    "Step 1: Get patient record",
    "[TOOL]",
    get_patient_record,
    patient_id
)

reason = validate_patient_inputs(patient_record)
if reason:
    review = run_traced(
        "Step 0: Input validation escalation",
        "[TOOL]",
        flag_for_manual_review,
        patient_id,
        reason
    )
    return finish(review)


In [ ]:
signoff = run_traced(
    "Step 2: Check doctor sign-off",
    "[TOOL]",
    get_doctor_signoff_status,
    patient_id
)

notes = run_traced(
    "Step 3: Get doctor notes",
    "[TOOL]",
    get_doctor_notes,
    patient_id
)

In [ ]:
extraction = run_traced(
    "Step 4: Extract diagnosis and treatment",
    "[AGENT]",
    extract_with_one_retry,
    notes
)

if not extraction.get("icd_code") or not extraction.get("cpt_code"):
    review = run_traced(
        "Escalate unresolved codes",
        "[TOOL]",
        flag_for_manual_review,
        patient_id,
        "Diagnosis or procedure code unresolved after one retry"
    )
    return finish(review)
covered_amount = run_traced(
    "Step 9: Calculate covered amount",
    "[TOOL]",
    calculate_covered_amount,
    policy,
    total_bill,
    patient_record["days_admitted"],
    eligibility,
    bill_items=bill_items
)

In [ ]:
from collections import Counter

batch_results = []

for patient_id in PATIENTS:
    print(f"\n{'=' * 50}")
    print(f"Processing patient: {patient_id}")
    print("=" * 50)

    try:
        result = orchestrate_claim_process(patient_id, verbose=False)

        # Adjust these checks if your functions use different status labels.
        status = str(result.get("status", "")).lower()
        reason = (
            result.get("reason")
            or result.get("review_reason")
            or result.get("message")
            or ""
        )

        if "review" in status or "escalat" in status:
            outcome = "Escalated"
        elif status in {"completed", "complete", "success", "approved"}:
            outcome = "Completed"
        elif result.get("covered_amount") is not None:
            outcome = "Completed"
        else:
            outcome = "Check result"

        batch_results.append({
            "patient_id": patient_id,
            "outcome": outcome,
            "reason": reason,
        })

    except Exception as exc:
        batch_results.append({
            "patient_id": patient_id,
            "outcome": "Error",
            "reason": f"{type(exc).__name__}: {exc}",
        })

summary_df = pd.DataFrame(batch_results)

print("\nBatch summary")
print(summary_df.to_string(index=False))

print("\nOutcome counts")
print(summary_df["outcome"].value_counts().to_string())

print("\nEscalation reasons")
escalated = summary_df[summary_df["outcome"] == "Escalated"]
if escalated.empty:
    print("No escalations recorded.")
else:
    counts = Counter(
        row["reason"] or "Reason not returned"
        for _, row in escalated.iterrows()
    )
    for reason, count in counts.most_common():
        print(f"{count} patient(s): {reason}")